In [1]:
# 여기 코드 한줄한줄 아주 상세히 설명 좀 해줘. 왜 이런걸 썻고 왜 이런 문법을 썻고 이 함수가 뭐고 이 변수는 뭐고 그렇게

In [2]:
import torch
from transformers import AutoTokenizer, AutoModel
import pandas as pd
import pyarrow.parquet as pq
import numpy as np
from tqdm import tqdm

In [3]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

"""
HuggingFace 모델이랑 토크나이저 같은거 다 불러오자
참고로 tokenizer은.. 문자를 모델의 입력으로 바꿔주는 도구래! 
미리 학습되어있는거 불러오면 좋겠지?
"""
# model_name = "intfloat/e5-small"
model_name = "intfloat/e5-base-v2"
tokenizer = AutoTokenizer.from_pretrained(model_name) # model_name 넣어주면 알아서 불러와줌
model = AutoModel.from_pretrained(model_name) # .to(device)
lora_config = LoraConfig(
    task_type=TaskType.FEATURE_EXTRACTION,
    r=16,
    lora_alpha=32,
    lora_dropout=0.1,
    bias="none"
)
model = get_peft_model(model, lora_config).to(device)

# 저장된 state_dict 불러오기
model.load_state_dict(torch.load('e5-lora-kd.pth'))
model.eval()

BertModel(
  (embeddings): BertEmbeddings(
    (word_embeddings): Embedding(30522, 384, padding_idx=0)
    (position_embeddings): Embedding(512, 384)
    (token_type_embeddings): Embedding(2, 384)
    (LayerNorm): LayerNorm((384,), eps=1e-12, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder): BertEncoder(
    (layer): ModuleList(
      (0-11): 12 x BertLayer(
        (attention): BertAttention(
          (self): BertSdpaSelfAttention(
            (query): Linear(in_features=384, out_features=384, bias=True)
            (key): Linear(in_features=384, out_features=384, bias=True)
            (value): Linear(in_features=384, out_features=384, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (output): BertSelfOutput(
            (dense): Linear(in_features=384, out_features=384, bias=True)
            (LayerNorm): LayerNorm((384,), eps=1e-12, elementwise_affine=True)
            (dropout): Dropout(p=0.1, inplace=False

In [4]:
DATA_PATH = './data/book_meta.parquet'
df = pd.read_parquet(DATA_PATH)
df = df.dropna(subset=["title", "description"]) # 결측치 제거
df = df.reset_index(drop=True) # 기존 index 초기화, drop=true하면 기존 index를 컬럼으로 안만들고 걍 제거함

# 입력 텍스트 생성 (타이틀 + 설명 + 저자 등 결합)
def build_text(row):
    parts = [
        f"Title: {row['title']}",
        # f"Author: {row['authors']}",
        # f"Publisher: {row['publisher']}",
        f"Description: {row['description']}"
    ]
    return " ".join( # 리스트의 문자열들을 공백으로 연결할건데.....
        [p for p in parts if isinstance(p, str)] # NaN이나 None이 있으면 제외함
    ) # 최종적으로 하나의 문장 형태로 반환한다고 함!! "Title: ... Author: ... Publisher: ... Description: ..."

df["text"] = df.apply(build_text, axis=1) # 새 컬럼 text에 대해서.... 문장 만듦

In [5]:
from datasets import Dataset
import numpy as np

# Hugging Face의 datasets는 메모리 대신 디스크 캐시 기반
data = {"book_id": df["book_id"], "isbn": df["isbn"], "text": df["text"]}
dataset = Dataset.from_dict(data)

"""
embed_batch 함수:
책 텍스트 리스트를 받아 E5 임베딩을 batch 단위로 만들어 반환함
얘는 대량으로 처리하기 좋은 방법이라서..... 뭐 정기적으로 대량임베딩하거나 할때 좋대
나중에 e5를 다른걸로 바꾸거나 뭐.... 모델 운영하다보면 semantic search가 부정확해지거나 할때 쓰면 굿
"""
def embed_batch(batch):
    batch_texts = [f"passage: {t}" for t in batch["text"]] # 각 텍스트 앞에 passage:를 붙힘!(e5 권장; 문맥 signal)

    inputs = tokenizer(
        batch_texts, return_tensors="pt", 
        truncation=True, # 길이 넘치면 자름
        padding=True,  # batch 단위로 다른 문장 길이에 맞게 padding
        max_length=256 # 최대 토큰 길이 제한
    ).to(device)

    with torch.no_grad():  # gradient 비활성화
        outputs = model(**inputs) # 딕셔너리를 언패킹(**)하여 모델에 전달
    
        emb = outputs.last_hidden_state.mean(dim=1)  # mean pooling
        # 원래는 아웃풋이 다 토큰단위인데.... mean해줘서 문장단위로 임베딩하게 된다는뎅
    
        emb = torch.nn.functional.normalize(emb, p=2, dim=1) # 정규화

    # pytorch 텐서는 기본적으로 연산 그래프를 추적해서 back-prop을 계산하나봐
    # 근데 .cpu().numpy()는 오직 gradient 추적 없는 순수 값(텐서)만 가능한 OP라서
    # .detach()를 통해서 그래프를 끊고 순수 값으로 탈바꿈 시킨대
    emb = emb.detach().cpu().numpy().tolist()
    return {"embedding": emb}

dataset = dataset.map(embed_batch, batched=True, batch_size=128)

# ========= 얘는 걍 제목이나 저자 그런것도 같이 parquet로 저장하는겨 =========
# dataset.to_parquet("book_emb.parquet")

# ==================== 얘는 id랑 임베딩된 값만 저장하는겨 ====================
book_emb_dict = { 
    int(row["book_id"]): np.array(row["embedding"], dtype=np.float32)
    for row in dataset # dict로 변환 (book_id -> np.array)
}
np.save("book_emb_dict.npy", book_emb_dict) # numpy로 저장
print("✅ book_emb_dict.npy 저장 완료!")

Map:   0%|          | 0/2260655 [00:00<?, ? examples/s]

✅ book_emb_dict.npy 저장 완료!


In [6]:
"""
e5_encode 함수
책 텍스트 리스트를 받아 E5 임베딩을 batch 단위로 만들어 반환함
근데 얘.... np.vstack으로 저장하는거래 대용량으로 뭐 .csv로 바꿀때는 램 터짐
그래도 실시간으로 처리해야하는 시스템이선 쓰면 좋대 이렇게
"""
# def e5_encode(texts, batch_size=16):
#     embeddings = []
#     with torch.no_grad(): # gradient 계산 비활성화
#         for i in tqdm(range(0, len(texts), batch_size)): # batch_size 사이즈 간격으로!
            
#             batch_texts = [ # 각 텍스트 앞에 passage:를 붙힘!(e5 권장; 문맥 signal)
#                 f"passage: {t}" for t in texts[i:i+batch_size]
#             ]

#             inputs = tokenizer(
#                 batch_texts, return_tensors="pt", 
#                 truncation=True, # 길이 넘치면 자름
#                 padding=True,  # batch 단위로 다른 문장 길이에 맞게 padding
#                 max_length=256 # 최대 토큰 길이 제한
#             ).to(device)
            
#             outputs = model(**inputs) # 딕셔너리를 언패킹(**)하여 모델에 전달
            
#             emb = outputs.last_hidden_state.mean(dim=1)  # mean pooling
#             # 원래는 아웃풋이 다 토큰단위인데.... mean해줘서 문장단위로 임베딩하게 된다는뎅
            
#             emb = torch.nn.functional.normalize(emb, p=2, dim=1) # 정규화
#             embeddings.append(emb.cpu().numpy())
#     return np.vstack(embeddings)

# book_embeddings = e5_encode(df["text"].tolist(), batch_size=64)

'\ne5_encode 함수\n책 텍스트 리스트를 받아 E5 임베딩을 batch 단위로 만들어 반환함\n근데 얘.... np.vstack으로 저장하는거래 대용량으로 뭐 .csv로 바꿀때는 램 터짐\n그래도 실시간으로 처리해야하는 시스템이선 쓰면 좋대 이렇게\n'